In [1]:
import numpy as np
import time
from gridcp.new_api.detector import GridDetector, DetectorState
from gridcp.new_api.scores import MeanCUSUM
from gridcp.new_api.typing import ArrayLike
import gridcp

In [2]:
def run_online_grid_detector(
    data: ArrayLike,
    detector: GridDetector,
    reset_on_alarm: bool = False,
) -> tuple[DetectorState, dict]:
    """Run a configured GridDetector over a dataset sequentially.

    Parameters
    ----------
    data : ArrayLike
        Sequence of observations. Each element is passed as `x` to
        `detector.update`. For univariate data this is a 1D array of scalars;
        for multivariate data it should be a 2D array of shape (n_samples, n_features).
    detector : GridDetector
        A fully configured detector instance (with `score` and `threshold`
        set). State is initialised internally via `detector.init_state()`.
    reset_on_alarm : bool, optional
        If True, the detector state is reset to a fresh initial state immediately
        after an alarm is raised. This allows the detector to restart tracking
        from the next observation after each detected changepoint.
        Default is False.

    Returns
    -------
    state : DetectorState
        State after processing the full input sequence.
    output : dict
        Output dictionary from the final observation. It contains:
          - ``"index"``: time index (n_samples) after update.
          - ``"alarm"``: bool, True when ``max_score > threshold``.
          - ``"max_score"``: highest penalised score among active candidates.
          - ``"max_score_index"``: grid position of the highest-scoring candidate.
    """
    data = np.asarray(data)
    state = detector.init_state()
    output = None
    for x in data:
        state, output = detector.update(state, x)

    return state, output

In [3]:
def demo(n_samples=100, n_features=1, N=100, reset_on_alarm=False):
    """Simulate toy univariate data with a known mean shift and run the grid detector."""
    rng = np.random.default_rng(seed=42)
    finalmaxx = []
    for i in range(N):
        n_pre, n_post = n_samples // 2, n_samples // 2 + n_samples % 2
        data = np.concatenate(
            [
                rng.normal(loc=0.0, scale=1.0, size=(n_pre, n_features)),
                rng.normal(loc=0.0, scale=1.0, size=(n_post, n_features)),
            ]
        )

        score = MeanCUSUM(n_features)
        detector = GridDetector(score=score, threshold=1000.0)
        state, output = run_online_grid_detector(data, detector, reset_on_alarm)
        finalmaxx.append(output["max_score"])

In [4]:
n_samples = 1000
N = 500

In [11]:
start = time.perf_counter()
demo(n_samples=n_samples, N=N, reset_on_alarm=False)
stop = time.perf_counter()
print(f"Execution time: {stop - start:.4f} seconds")

Execution time: 2.9101 seconds


In [ ]:
detector = gridcp.make_univariate_mean_change_detector()
start = time.perf_counter()
detector.calibrate_false_alarm(alpha=0.05, N=n_samples, K=N, null_dist=np.random.normal)
stop = time.perf_counter()
print(f"Execution time: {stop - start:.4f} seconds")

Execution time: 0.4247 seconds


In [ ]:
np.random.seed(42)
n_samples = 500
n_pre = n_samples // 2
data = np.concatenate(
    [
        np.random.normal(loc=0.0, scale=1.0, size=n_pre),
        np.random.normal(loc=0.0, scale=1.0, size=n_samples - n_pre),
    ]
)

old_detector = gridcp.make_univariate_mean_change_detector(penalty_constant=10.0)
new_score = MeanCUSUM(n_features=1)
new_detector = GridDetector(score=new_score, threshold=10.0)
state = new_detector.init_state()
output = None

for x in data:
    old_detector.update(x)
    state, output = new_detector.update(state, x)
    assert state.grid == [
        -g - 1 for g in old_detector._state["grid_list"]
    ], "Grid lists do not match between old and new detectors."

    assert np.isclose(
        state.running_score_state.sum, old_detector._state["sum"]
    ), "Score sums do not match between old and new detectors."

    for x, y in zip(old_detector._state["sum_pre_list"], state.candidate_score_states):
        assert np.isclose(
            x, y.sum
        ), "Sum pre lists do not match between old and new detectors."

In [30]:
# Unknown-variance univariate mean-change: old API vs new API equivalence check
import numpy as np
import gridcp
from gridcp.new_api.detector import GridDetector as NewGridDetector
from gridcp.scores import MeanCUSUMUnknownVariance

np.random.seed(42)
n_samples = 500
n_pre = n_samples // 2
data = np.concatenate(
    [
        np.random.normal(loc=0.0, scale=1.0, size=n_pre),
        np.random.normal(loc=0.0, scale=1.0, size=n_samples - n_pre),
    ]
)

old_detector = gridcp.make_univariate_mean_change_detector(
    penalty_constant=10.0, mode="unknown_variance"
)
new_score = MeanCUSUMUnknownVariance()
new_detector = NewGridDetector(score=new_score, threshold=10.0)
state = new_detector.init_state()
output = None

for x in data:
    old_detector.update(x)
    state, output = new_detector.update(state, np.asarray([x]))

    assert state.grid == [
        -g - 1 for g in old_detector._state["grid_list"]
    ], "Grid lists do not match between old and new detectors (unknown variance)."

    assert np.allclose(
        state.running_score_state.stats, old_detector._state["sum"]
    ), "Running sufficient statistics do not match (unknown variance)."

    for old_stats, new_stats in zip(
        old_detector._state["sum_pre_list"], state.candidate_score_states
    ):
        assert np.allclose(
            old_stats, new_stats.stats
        ), "Prefix sufficient statistics do not match (unknown variance)."

# Add one extreme final observation and compare max score statistics directly.
x_extreme = 1_000_000.0
old_alarm = old_detector.update(x_extreme)
state, output = new_detector.update(state, np.asarray([x_extreme]))

assert (
    old_alarm and output["alarm"]
), "Both detectors should alarm on extreme observation."
assert np.isclose(
    old_detector.max_statistic, output["max_score"]
), f"Max score mismatch: old={old_detector.max_statistic}, new={output['max_score']}"

print("Unknown-variance old/new API checks passed, including extreme-point max score.")

Unknown-variance old/new API checks passed, including extreme-point max score.
